# The alpha dial as places, not scores

Where each dial position lands **in the ranking**, on validation, for both boosting
heads. One figure, two panels: `cls+mol` on the left --- the construction the paper
reports, where the refined receptor REPLACES the protein vector --- and
`cls+prot+mol` on the right, where it is added beside it.

Why ranks and not the metric itself: a cell is (dataset, regime, molecule source), and
$R^2$ on Carey and AUROC on M2OR cannot be averaged. Their orderings can. So every
competitor is ranked *within* a cell first, and what is plotted is the mean of those
places across cells, with the spread across cells as the bar.

**This notebook draws and tests; it does not invent an aggregation.** The ranks come
from `scripts/analysis/alpha_choice.py`, the same module the (unused) table script
reads, so the figure and that script cannot disagree. Model seeds are averaged inside
each held-out split before anything else, so a cell's score rests on splits and not on
(split, seed) pairs.

**The question this page exists to answer.** Not "which alpha wins" --- an argmax over
a flat surface is noise --- but *is the surface flat at all*. The fitted line is that
question drawn, and the cell below it is that question tested.

The choice table (`scripts/legacy/06_alpha_choice_not_used.py`) is **not** part
of the paper as of 22.09. It still runs and still implements the protocol; this figure
replaces it.

In [ ]:
import pathlib
import sys
import types

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks" / "article_figures"))

import figkit as fk
from scripts.analysis import alpha_grid as ag
from scripts.analysis import alpha_choice as ac

fk.use_house_style()
print("repo:", ROOT)

## Knobs

In [ ]:
# ================================== KNOBS ====================================
ROOT_DIR   = "results/graph/v13_esm3"   # the sweep this figure is read from
NODES      = "nodedial"                 # "nodedial" (v9 node dial) | "onehot" (v8 gate)
MOL_SOURCE = "chemberta"                # the dense dial grid exists ONLY here -- see below
DATASET    = None          # None -> every dataset on disk; or ["cc", "hc"] / "m2or"
REGIME     = None          # None -> both; or "transductive" / "inductive"
SPLIT      = "val"         # alpha is a hyperparameter: it is looked at on VALIDATION
HEADS      = ("cls+mol", "cls+prot+mol")   # left panel, right panel
METRIC     = None          # None -> the metric of record per dataset

REFERENCE  = "boost"       # the dashed horizontal line: where the base ranks
FIT        = True          # draw the least-squares line through the dial positions
N_PERM     = 20000         # permutations for the slope test in the last cell
PERM_SEED  = 0

SAVE_FIGS  = False
FIG_DIR    = "results/figures/alpha_rank_dial"
# =============================================================================

# MOL_SOURCE is not cosmetic. The dense grid was run on chemberta only; on gin and ecfp
# just alpha in {0, 1} exists. Mean ranks pooled over cells holding 13 competitors and
# cells holding 3 are not on one scale -- the middle of a crowded cell is rank 7, the
# middle of a sparse one is rank 2. The coverage check below refuses to let that pass
# silently.

## The ranks

One frame per head: every competitor (each dial position, plus the boosting base and
the legacy graph) ranked within each cell, then summarised across cells.

In [ ]:
def cell_ranks(head):
    """Per (cell, competitor): the place, 1 = best. Cells are the sample."""
    df = ag.load(root=ROOT_DIR, nodes=NODES, mol_source=MOL_SOURCE,
                 dataset=DATASET, regime=REGIME, combo=head, split=SPLIT)
    metric_of = ac._metric_of(df, types.SimpleNamespace(metric=METRIC))
    sc = ac.scores(df, metric_of)
    # ranked DESCENDING because every metric of record is higher-is-better; ties take
    # the average place, so two identical arms do not split a place by row order
    return sc.assign(rank=sc.groupby(ac.CELL)["value"]
                     .rank(ascending=False, method="average"))


def summarise(sc):
    """Mean place and its spread across cells, per competitor."""
    g = sc.groupby(["competitor", "alpha"], dropna=False)["rank"]
    out = g.agg(mean="mean", std=lambda s: s.std(ddof=1), cells="size").reset_index()
    return out.sort_values("mean").reset_index(drop=True)


RANKS = {h: cell_ranks(h) for h in HEADS}
SUMMARY = {h: summarise(r) for h, r in RANKS.items()}

for h in HEADS:
    s = SUMMARY[h]
    counts = sorted(s["cells"].unique())
    print(f"{h}: {len(s)} competitors, {int(s['cells'].max())} cells")
    if len(counts) > 1:
        print(f"  !! RAGGED: competitors cover {counts} cells. A mean place taken over "
              f"different cell sets is not a comparison -- set MOL_SOURCE to the one "
              f"the dense grid was run on, or read this panel as decoration.")
SUMMARY[HEADS[0]].head(20)

## The figure

Height is the mean place across cells, so **lower is better** and the y axis is left in
its natural direction rather than flipped --- a flipped axis makes the sign of the
fitted slope unreadable, and the slope is the point.

The bar is the standard deviation across cells: how much the panels disagree about
where this dial position belongs. It is not a confidence interval and is not meant to
be read as one.

In [ ]:
def fit_line(s):
    """Least squares through the dial positions. Returns (slope, intercept, fn)."""
    g = s[s["alpha"].notna()]
    if len(g) < 3:
        return None
    r = stats.linregress(g["alpha"].astype(float), g["mean"].astype(float))
    return r, (lambda x: r.intercept + r.slope * np.asarray(x, float))


def panel(ax, head):
    s = SUMMARY[head]
    g = s[s["alpha"].notna()].sort_values("alpha")
    if g.empty:
        fk.note_empty(ax, "no dial arm")
        return
    ax.errorbar(g["alpha"], g["mean"], yerr=g["std"], fmt="o", color=fk.C.get("gate", "#0072B2"),
                capsize=2.5, lw=0, elinewidth=1.1, markersize=4.5, zorder=3)

    ref = s[s["competitor"] == REFERENCE]
    if len(ref):
        fk.reference_line(ax, float(ref.iloc[0]["mean"]), "boost_full")

    if FIT:
        got = fit_line(s)
        if got:
            r, f = got
            xs = np.linspace(0, 1, 50)
            ax.plot(xs, f(xs), color=fk.MUTED, lw=1.4, zorder=2)
            ax.set_title(f"{head}\nslope {r.slope:+.2f} places per unit "
                         f"$\\alpha$  (p = {r.pvalue:.2f})")
        else:
            ax.set_title(head)
    else:
        ax.set_title(head)

    fk.alpha_axis(ax, sorted(g["alpha"].unique()))
    ax.set_xlabel(r"$\alpha$  (0 = identity nodes, 1 = ESM nodes)")


fig, axes = fk.panels(len(HEADS), ncols=2, w=3.6, h=3.0)
for ax, head in zip(axes, HEADS):
    panel(ax, head)
axes[0].set_ylabel("mean place across cells  (1 = best)")

handles = [Line2D([], [], color=fk.C.get("gate", "#0072B2"), marker="o", lw=0,
                  label="dial position (bar: SD across cells)"),
           Line2D([], [], color=fk.C.get("boost_full", fk.MUTED), lw=1.3,
                  ls=fk.DASH["boost_full"], label="boosting base"),
           Line2D([], [], color=fk.MUTED, lw=1.4, label="least-squares fit")]
fk.figlegend(fig, handles, ncol=3)
fk.savefig(fig, "alpha_rank_dial", FIG_DIR, SAVE_FIGS)
plt.show()

## Is that slope zero?

Three tests of one null, in increasing order of how much they should be believed.

1. **Fit through the dial positions.** The line the figure draws, with the p-value
   `linregress` reports. Descriptive only: the points are mean places from the *same*
   cells, and inside a cell the places are ranked against each other, so they sum to a
   constant and are not independent. Quoting this p-value as the result would be the
   same mistake as counting (fold, seed) pairs as observations.
2. **A fit per cell, the cells being the sample.** One slope per cell, then a
   one-sample t-test of their mean against zero. This is the unit of evidence used
   everywhere else in this project, and it is the number to report.
3. **A permutation test.** The dial labels are shuffled *within* each cell, which
   destroys any relation to alpha while preserving exactly the rank structure the cell
   imposes. The p-value is the share of shuffles whose mean slope is at least as far
   from zero as the observed one. It assumes nothing about normality, and with a
   handful of cells that matters.

In [ ]:
def per_cell_slopes(sc):
    """One least-squares slope of place on alpha per cell."""
    rows = []
    for key, g in sc.groupby(ac.CELL, sort=True):
        g = g[g["alpha"].notna()]
        if g["alpha"].nunique() < 3:
            continue
        r = stats.linregress(g["alpha"].astype(float), g["rank"].astype(float))
        rho, _ = stats.spearmanr(g["alpha"].astype(float), g["rank"].astype(float))
        rows.append(dict(zip(ac.CELL, key))
                    | dict(slope=float(r.slope), rho=float(rho), points=len(g)))
    return pd.DataFrame(rows)


def permuted_p(sc, observed, n_perm=N_PERM, seed=PERM_SEED):
    """Shuffle the dial labels inside each cell; how often is the mean slope this big?"""
    rng = np.random.default_rng(seed)
    cells = [g[g["alpha"].notna()] for _, g in sc.groupby(ac.CELL, sort=True)]
    cells = [g for g in cells if g["alpha"].nunique() >= 3]
    if not cells:
        return np.nan
    xs = [g["alpha"].astype(float).to_numpy() for g in cells]
    ys = [g["rank"].astype(float).to_numpy() for g in cells]
    hits = 0
    for _ in range(n_perm):
        slopes = [stats.linregress(x, rng.permutation(y)).slope
                  for x, y in zip(xs, ys)]
        hits += abs(np.mean(slopes)) >= abs(observed) - 1e-12
    return (hits + 1) / (n_perm + 1)      # +1: a permutation p is never exactly 0


report = []
for head in HEADS:
    s, sc = SUMMARY[head], RANKS[head]
    got = fit_line(s)
    per = per_cell_slopes(sc)
    if got is None or per.empty:
        report.append(dict(head=head, note="too few dial positions"))
        continue
    r, _ = got
    mu, hw, n = ag.ci(per["slope"])
    t, p_t = stats.ttest_1samp(per["slope"].to_numpy(), 0.0)
    p_perm = permuted_p(sc, float(mu))
    report.append(dict(
        head=head, cells=int(n),
        fit_slope=round(float(r.slope), 3), fit_p=round(float(r.pvalue), 4),
        cell_slope=round(float(mu), 3),
        cell_ci=f"[{mu - hw:+.2f}, {mu + hw:+.2f}]",
        cell_t=round(float(t), 2), cell_p=round(float(p_t), 4),
        perm_p=round(float(p_perm), 4),
        verdict=("flat: zero is inside the interval" if (mu - hw) <= 0 <= (mu + hw)
                 else "sloped: zero is outside the interval")))

print("slope is in PLACES per unit alpha: negative = the place improves (moves toward 1)")
print("as alpha rises.\n")
pd.DataFrame(report)

### Reading it

If the per-cell interval covers zero and the permutation p is unremarkable, the dial is
a **flat surface** and no position on it is better than another by this evidence. That
is a result, not a null finding: it is what licenses reporting $\alpha = 1$ --- the end
of the scale, the graph exactly as it appears everywhere else in this paper --- rather
than whichever position happened to come out on top.

If the slope is real, say so and say which way it points; a dial that genuinely
improves as ESM is mixed in is a different paper from a dial that does not.

The per-cell frame is worth a look either way: if the cells disagree in sign, the mean
slope is a summary of a disagreement, and the panels should be named in the text rather
than averaged into one sentence.

In [ ]:
pd.concat([per_cell_slopes(RANKS[h]).assign(head=h) for h in HEADS],
          ignore_index=True).round(3)